## Crossover sensitivity

**Calculate percentage of patients that are high risk at different crossover thresholds: 14d, 30d, and 60dd**

In [1]:
import numpy as np
import pandas as pd

## Import data

In [2]:
treatment_df = pd.read_csv('../outputs/pembrochemo_pembro_index.csv')

In [3]:
treatment_df.sample(3)

,PatientID,LineName,StartDate
11872,F2B408C1F610D,pembro_platinum,2022-06-28
11115,FDF0498942499,pembro_platinum,2018-08-06
8540,F00D06902341A,pembro_platinum,2022-12-12


In [4]:
treatment_df.shape

(20623, 3)

In [5]:
treatment_df['treatment'] = (treatment_df['LineName'] == 'pembro_platinum').astype(int)

In [6]:
dtype_map = pd.read_csv('../outputs/pembrochemo_pembro_features_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
features_df = pd.read_csv('../outputs/pembrochemo_pembro_features_df.csv', dtype = dtype_map)

In [7]:
features_df.shape

(2064, 164)

In [8]:
surv_pred_df = pd.read_csv('../outputs/gb_6m_survival_predictions_calibrated.csv')

In [9]:
surv_pred_df.head(3)

,PatientID,psurv_180_calibrated
0,FF16F972863F8,0.463395
1,F5EF114860555,0.578838
2,F95B93B796545,0.942137


In [10]:
df = pd.merge(features_df, treatment_df, on = 'PatientID', how = 'left')

In [11]:
df.shape

(2064, 167)

In [12]:
df = pd.merge(df, surv_pred_df, on = 'PatientID', how = 'left')

In [13]:
df.shape

(2064, 168)

In [14]:
df['StartDate'] = pd.to_datetime(df['StartDate'])

In [15]:
df['treatment_year'] = df['StartDate'].dt.year

In [16]:
df = df.query('treatment_year <= 2023')

In [17]:
df.shape

(1629, 169)

In [18]:
with open('../outputs/crossover_survival_estimate.txt', 'r') as f:
    crossover_survival_estimate_30 = float(f.read())

with open('../outputs/crossover_survival_estimate_14.txt', 'r') as f:
    crossover_survival_estimate_14 = float(f.read())

with open('../outputs/crossover_survival_estimate_60.txt', 'r') as f:
    crossover_survival_estimate_60 = float(f.read())

In [19]:
print(f'r* for 14d crossover: {crossover_survival_estimate_14}')
print(f'r* for 30d crossover: {crossover_survival_estimate_30}')
print(f'r* for 60d crossover: {crossover_survival_estimate_60}')

r* for 14d crossover: 0.8465270296691438
r* for 30d crossover: 0.7347847004756852
r* for 60d crossover: 0.5252678332379505


In [20]:
print(f'percent high risk at 14d crossover: {df.query('psurv_180_calibrated < @crossover_survival_estimate_14').shape[0]/df.shape[0]}')
print(f'percent high risk at 30d crossover: {df.query('psurv_180_calibrated < @crossover_survival_estimate_30').shape[0]/df.shape[0]}')
print(f'percent high risk at 60d crossover: {df.query('psurv_180_calibrated < @crossover_survival_estimate_60').shape[0]/df.shape[0]}')

percent high risk at 14d crossover: 0.6482504604051565
percent high risk at 30d crossover: 0.4014732965009208
percent high risk at 60d crossover: 0.14426028238182934
